In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import h3
import plotly.express as px 

os.environ['HAVEN_DATABASE'] = 'haven'
os.environ['AWS_PROFILE'] = 'admin'

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import plot_h3_slider

In [ ]:
sql = '''
select 
    origin_h3_index,
    next_h3_index,
    time,
    probability,
    stay_put
from 
    movement_model_full_inference_10_1_10
where 
    time = TIMESTAMP '2022-03-15 12:00:00'
'''
data = read_data_w_cache(sql)
print(data.shape)
data['lat'] = data['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
data['lon'] = data['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])
data = data[
    (data['lat'] > 47) & (data['lat'] < 65)
    & (data['lon'] < -120) & (data['lat'] > -176)
]
origins = sorted(data['origin_h3_index'].unique())
data = data[data['next_h3_index'].isin(origins)]
data['probability'] = data['probability'].fillna(0) + 0.00001
data['total_probability'] = data.groupby('origin_h3_index')['probability'].transform('sum')
data['probability'] = data['probability'] / data['total_probability']
print(data.shape)
data.head()

In [ ]:
data[data['origin_h3_index'] == origins[1]]

In [ ]:
indices = {
    h3_index: i 
    for i, h3_index in enumerate(origins)
}
len(indices)

In [ ]:
num_options = data.groupby('origin_h3_index').size().to_dict()

In [ ]:
M = np.zeros((len(indices), len(indices)))
N = np.zeros((len(indices), len(indices)))

for _, row in tqdm(data.iterrows()):
    num_neighors = num_options[row['origin_h3_index']]
    i = indices[row['next_h3_index']]
    j = indices[row['origin_h3_index']]
    M[i, j] = row['probability']
    M[i, j] = row['probability']
    N[i, j] = 1 / num_neighors

In [ ]:
MX = np.linalg.matrix_power(M, 7)
NX = np.linalg.matrix_power(N, 7)

In [ ]:
rows = []
for h3_index, i in indices.items():
    P = MX[:, i]
    Q = NX[:, i]
    P = P[Q != 0]
    Q = Q[Q != 0]
    divergence = sum(P * np.log(P/Q)) 
    rows.append({
        'h3_index': h3_index,
        'divergence': divergence,
    })
divergence = pd.DataFrame(rows)
print(divergence.shape)
divergence.head()

In [ ]:
divergence['version'] = 1
plot_h3_slider(divergence, 'divergence', 'h3_index', 'version', zmin=0, zmax=divergence['divergence'].quantile(0.9)).show()

In [ ]:
reverse_index = {
    i: h3_index 
    for h3_index, i in indices.items()
}

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'shift': p-q#p * np.log(p/q)#p - q
            })
shifts = pd.DataFrame(rows)
print(shifts.shape)
shifts.head()

In [ ]:
h3_index = '840cd99ffffffff'#'8422811ffffffff'#'84228a3ffffffff'
shifts['color'] = shifts.apply(lambda r: 'red' if r['origin_h3_index'] == r['h3_index'] else 'blue', axis=1)
df = shifts[shifts['origin_h3_index'] == h3_index]
boundary = max(abs(df['shift'].min()), df['shift'].max())
plot_h3_slider(
    shifts[shifts['origin_h3_index'] == h3_index], 'shift', 'h3_index', 'origin_h3_index', line_color_col='color', bold_colors=['red'],
    colorscale='RdBu', zmin=-boundary, zmax=boundary
)

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    origin = h3_index
    olat, olon = h3.h3_to_geo(origin)
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            dest = reverse_index[j]
            dlat, dlon = h3.h3_to_geo(dest)
            NS = dlat - olat
            EW = dlon - olon
            length = (NS ** 2 + EW ** 2) ** 0.5
            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'N': NS, #/ length if length != 0 else 0,
                'E': EW, #/ length if length != 0 else 0,
                'shift': p
            })
shifts = pd.DataFrame(rows)
print(shifts.shape)
shifts.head()

In [ ]:
angle = np.pi/2
E = np.cos(angle)
N = np.sin(angle)

shifts['contribution'] = (shifts['N'] * N + shifts['E'] * E) * shifts['shift']
df = shifts.groupby('origin_h3_index')[['contribution', 'shift']].sum().reset_index()
df['version'] = 1
boundary = max(abs(df['contribution'].quantile(0.1)), abs(df['contribution'].quantile(0.9)))
plot_h3_slider(df, 'contribution', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu')

In [ ]:
angle = 0
E = np.cos(angle)
N = np.sin(angle)

shifts['contribution'] = (shifts['N'] * N + shifts['E'] * E) * shifts['shift']
df = shifts.groupby('origin_h3_index')[['contribution', 'shift']].sum().reset_index()
df['version'] = 1
boundary = max(abs(df['contribution'].quantile(0.1)), abs(df['contribution'].quantile(0.9)))
plot_h3_slider(df, 'contribution', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu')

In [ ]:
rows = [
    {
        'h3_index': h3_index,
        'stickiness': MX[i, i]
    }
    for h3_index, i in indices.items()
]
df = pd.DataFrame(rows)
df['version'] = 1
plot_h3_slider(
    df, 'stickiness', 'h3_index', 'version', zmin=df['stickiness'].quantile(0.5), zmax=df['stickiness'].quantile(0.9)
)

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    origin = h3_index
    olat, olon = h3.h3_to_geo(origin)
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            dest = reverse_index[j]
            dlat, dlon = h3.h3_to_geo(dest)
            NS = dlat - olat
            EW = dlon - olon
            length = (NS ** 2 + EW ** 2) ** 0.5
            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'N': NS, #/ length if length != 0 else 0,
                'E': EW, #/ length if length != 0 else 0,
                'p': p,
                'q': q
            })
shifts = pd.DataFrame(rows)
print(shifts.shape)
shifts.head()

In [ ]:
shifts['q_N'] = shifts['N'] * shifts['q'] 
shifts['q_E'] = shifts['E'] * shifts['q'] 
shifts['p_N'] = shifts['N'] * shifts['p'] 
shifts['p_E'] = shifts['E'] * shifts['p'] 
df = shifts.groupby('origin_h3_index')[['q_N', 'q_E', 'q', 'p_N', 'p_E', 'p']].sum().reset_index()
df['q_N'] = df['q_N'] / df['q']
df['q_E'] = df['q_E'] / df['q']
df['q_anisotropy'] = (df['q_N'] ** 2 + df['q_E'] ** 2) ** 0.5
df['p_N'] = df['p_N'] / df['p']
df['p_E'] = df['p_E'] / df['p']
df['p_anisotropy'] = (df['p_N'] ** 2 + df['p_E'] ** 2) ** 0.5


In [ ]:
df['version'] = 1
df['anisotropy'] = df['p_anisotropy'] - df['q_anisotropy']
plot_h3_slider(
    df, 'anisotropy', 'origin_h3_index', 'version', zmin=df['anisotropy'].quantile(0.1), zmax=df['anisotropy'].quantile(0.9)
)

In [ ]:
angle = np.pi/2
E = np.cos(angle)
N = np.sin(angle)
df['value'] = df['p_E'] * E + df['p_N'] * N
boundary = max(abs(df['value'].quantile(0.1)), abs(df['value'].quantile(0.9)))
plot_h3_slider(
    df, 'value', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu'
)

In [ ]:
angle = 0
E = np.cos(angle)
N = np.sin(angle)
df['value'] = df['p_E'] * E + df['p_N'] * N
boundary = max(abs(df['value'].quantile(0.1)), abs(df['value'].quantile(0.9)))
plot_h3_slider(
    df, 'value', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu'
)